In [ ]:
import keras
import tensorflow as tf
from time import strftime
from pathlib import Path
from os import mkdir

def lr_schedule(epoch, lr):
    base_lr=0.01
    if epoch<4:
        return base_lr
    elif epoch<8:
        return base_lr*0.5
    elif epoch<=10:
        return base_lr*0.025

def get_run_log_dir(root_dir='my_logs'):
    root_path=Path(root_dir)
    root_path.mkdir(exist_ok=True,parents=True)
    return Path(root_dir)/strftime("run_%Y_%m_%d_%H_%M_%S")

run_dir=get_run_log_dir()

cifar=keras.datasets.cifar10.load_data()

(X_train,y_train),(X_test,y_test)=cifar

X_train,X_test=X_train/255.0,X_test/255.0

X_valid,y_valid=X_train[-5000:],y_train[-5000:]

X_train,y_train=X_train[:-5000],y_train[:-5000]

tf.random.set_seed(0)

elu_act=keras.activations.elu

he_init=keras.initializers.HeNormal()

l2_reg=keras.regularizers.l2()

# data_aug=keras.Sequential([
#     keras.layers.RandomFlip(mode='horizontal'),
#     keras.layers.RandomTranslation(0.1,0.1),
# ])

inputs=keras.layers.Input(shape=(32,32,3),name='input')

#x=data_aug(inputs)

x=keras.layers.Conv2D(32,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D(pool_size=(2,2))(x)

x=keras.layers.Conv2D(64,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_1=keras.layers.Conv2D(64,(1,1),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_1_output)
x=keras.layers.add([x,skip_1])
x=keras.layers.MaxPool2D(2,2)(x)
block_2_output=x

x=keras.layers.Conv2D(128,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_2=keras.layers.Conv2D(128,(1,1),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_2_output)
x=keras.layers.add([x,skip_2])
x=keras.layers.MaxPool2D(2,2)(x)
block_3_output=x

flatten=keras.layers.GlobalMaxPooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(flatten)
outputs=keras.layers.Dense(10, activation='softmax')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.01, nesterov=True, momentum=0.9)

Adam_optimizer=keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)

loss=keras.losses.SparseCategoricalCrossentropy()

model.compile(optimizer=SGD_optimizer,loss=loss,metrics=['accuracy'])

tensorboard_cb=keras.callbacks.TensorBoard(run_dir)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor='val_accuracy',patience=7,verbose=1, restore_best_weights=True)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy',factor=0.05, patience=3,verbose=1, min_lr=1e-6)

LrScedule_cb=keras.callbacks.LearningRateScheduler(lr_schedule, verbose=1)

history=model.fit(
    X_train,y_train,
    epochs=10,
    verbose=1,
    validation_data=(X_valid,y_valid),
    batch_size=32,
    callbacks=[tensorboard_cb ,LrScedule_cb, earlyStop_cb]
)

#model.save("Saved Models/Residual_CNN_3x3_3ConvBlocks_ELU_SGD_BN_DO02_LearningRatePlateau_FlipSwitch.keras")

2026-01-19 16:39:04.919860: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-19 16:39:04.919956: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-19 16:39:04.919960: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-19 16:39:04.920123: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-19 16:39:04.920313: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



Epoch 1: LearningRateScheduler setting learning rate to 0.01.
Epoch 1/10


2026-01-19 16:39:07.500727: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1407/1407 ━━━━━━━━━━━━━━━━━━━━ 88s 60ms/step - accuracy: 0.4126 - loss: 4.9889 - val_accuracy: 0.3952 - val_loss: 2.2974 - learning_rate: 0.0100

Epoch 2: LearningRateScheduler setting learning rate to 0.01.
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 85s 60ms/step - accuracy: 0.5613 - loss: 1.7358 - val_accuracy: 0.4760 - val_loss: 2.0653 - learning_rate: 0.0100

Epoch 3: LearningRateScheduler setting learning rate to 0.01.
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 85s 60ms/step - accuracy: 0.5979 - loss: 1.6827 - val_accuracy: 0.4238 - val_loss: 2.1989 - learning_rate: 0.0100

Epoch 4: LearningRateScheduler setting learning rate to 0.01.
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 87s 62ms/step - accuracy: 0.6153 - loss: 1.6601 - val_accuracy: 0.4704 - val_loss: 2.1889 - learning_rate: 0.0100

Epoch 5: LearningRateScheduler setting learning rate to 0.005.
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 92s 65ms/step - accuracy: 0.6741 - loss: 1.3772 - val_accuracy: 0.5994 - val_loss: 1.5581 - 

In [ ]:
import keras
import tensorflow as tf
from time import strftime
from pathlib import Path
from os import mkdir

def lr_schedule(epoch, lr):
    base_lr=0.01
    if epoch<4:
        return base_lr
    elif epoch<8:
        return base_lr*0.5
    elif epoch<=10:
        return base_lr*0.025

def get_run_log_dir(root_dir='my_logs'):
    root_path=Path(root_dir)
    root_path.mkdir(exist_ok=True,parents=True)
    return Path(root_dir)/strftime("run_%Y_%m_%d_%H_%M_%S")

run_dir=get_run_log_dir()

cifar=keras.datasets.cifar10.load_data()

(X_train,y_train),(X_test,y_test)=cifar

X_train,X_test=X_train/255.0,X_test/255.0

X_valid,y_valid=X_train[-5000:],y_train[-5000:]

X_train,y_train=X_train[:-5000],y_train[:-5000]

tf.random.set_seed(0)

elu_act=keras.activations.elu

he_init=keras.initializers.HeNormal()

l2_reg=keras.regularizers.l2()

data_aug=keras.Sequential([
    keras.layers.RandomFlip(mode='horizontal'),
    keras.layers.RandomTranslation(0.1,0.1),
])

inputs=keras.layers.Input(shape=(32,32,3),name='input')

x=data_aug(inputs)

x=keras.layers.Conv2D(32,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(32,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
block_1_output=keras.layers.MaxPool2D(pool_size=(2,2))(x)

x=keras.layers.Conv2D(64,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_1_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(64,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_1=keras.layers.Conv2D(64,(1,1),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_1_output)
x=keras.layers.add([x,skip_1])
x=keras.layers.MaxPool2D(2,2)(x)
block_2_output=x

x=keras.layers.Conv2D(128,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_2_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
x=keras.layers.Conv2D(128,(3,3),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(activation=elu_act)(x)
x=keras.layers.Dropout(0.2)(x)
skip_2=keras.layers.Conv2D(128,(1,1),padding='same',kernel_regularizer=l2_reg, kernel_initializer=he_init)(block_2_output)
x=keras.layers.add([x,skip_2])
x=keras.layers.MaxPool2D(2,2)(x)
block_3_output=x

flatten=keras.layers.GlobalMaxPooling2D()(block_3_output)
x=keras.layers.Dense(128, activation=elu_act, kernel_initializer=he_init, kernel_regularizer=l2_reg)(flatten)
outputs=keras.layers.Dense(10, activation='softmax')(x)

model=keras.Model(inputs=inputs, outputs=outputs)

SGD_optimizer=keras.optimizers.SGD(learning_rate=0.01, nesterov=True, momentum=0.9)

Adam_optimizer=keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)

loss=keras.losses.SparseCategoricalCrossentropy()

model.compile(optimizer=SGD_optimizer,loss=loss,metrics=['accuracy'])

tensorboard_cb=keras.callbacks.TensorBoard(run_dir)

earlyStop_cb=keras.callbacks.EarlyStopping(monitor='val_accuracy',patience=7,verbose=1, restore_best_weights=True)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy',factor=0.5, patience=2,verbose=1, min_lr=1e-5)

LrScedule_cb=keras.callbacks.LearningRateScheduler(lr_schedule, verbose=1)

history=model.fit(
    X_train,y_train,
    epochs=40,
    verbose=1,
    validation_data=(X_valid,y_valid),
    batch_size=32,
    callbacks=[tensorboard_cb ,lrPlateau_cb, earlyStop_cb]
)

#model.save("Saved Models/Residual_CNN_3x3_3ConvBlocks_ELU_SGD_BN_DO02_LearningRatePlateau_FlipSwitch.keras")

Epoch 1/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 101s 63ms/step - accuracy: 0.3936 - loss: 4.8530 - val_accuracy: 0.4320 - val_loss: 2.0716 - learning_rate: 0.0100
Epoch 2/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 64ms/step - accuracy: 0.5319 - loss: 1.8031 - val_accuracy: 0.5030 - val_loss: 1.9085 - learning_rate: 0.0100
Epoch 3/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 88s 62ms/step - accuracy: 0.5666 - loss: 1.7465 - val_accuracy: 0.4354 - val_loss: 2.2066 - learning_rate: 0.0100
Epoch 4/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.5748 - loss: 1.7309
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 63ms/step - accuracy: 0.5780 - loss: 1.7198 - val_accuracy: 0.4536 - val_loss: 2.1181 - learning_rate: 0.0100
Epoch 5/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 90s 64ms/step - accuracy: 0.6390 - loss: 1.4595 - val_accuracy: 0.5418 - val_loss: 1.7694 - learning_rate: 0.0050
Epoch 6/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 90s 64ms/step - accuracy